#imports

In [1]:
import pandas as pd
import json
from pathlib import Path
import numpy as np

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer

#data


In [3]:
df_department = pd.read_csv("/content/department-v2.csv")
df_seniority = pd.read_csv("/content/seniority-v2.csv")

In [4]:
def workers_positions_to_csv(data):
    people = json.loads(Path(data).read_text(encoding="utf-8"))
    rows = []

    for person_idx, jobs in enumerate(people, start=1):
        jobs_sorted = sorted(jobs, key=lambda d: (d.get("startDate") is None, d.get("startDate")))
        for job_idx, j in enumerate(jobs_sorted, start=1):
            rows.append({
                "person_id": person_idx,
                "job_no": job_idx,
                "organization": j.get("organization"),
                "linkedin": j.get("linkedin"),
                "position": j.get("position"),
                "startDate": j.get("startDate"),
                "endDate": j.get("endDate"),
                "status": j.get("status"),
                "department": j.get("department"),
                "seniority": j.get("seniority"),
            })

    df = pd.DataFrame(rows, columns=[
        "person_id","job_no","organization","linkedin","position",
        "startDate","endDate","status","department","seniority"
    ])
    df["person_id"] = df["person_id"].astype(int)
    df["job_no"] = df["job_no"].astype(int)
    return df

In [5]:
df_t_annotated= workers_positions_to_csv("/content/linkedin-cvs-annotated.json")
df_t_notannotated= workers_positions_to_csv("/content//linkedin-cvs-not-annotated.json")

#Approach

### BoW / TF-IDF / Logistic Regression classification pipeline
- Each job title is represented as a sparse TF–IDF vector using:
  - CountVectorizer (of word n-grams, character n-grams)
  - TfidfTransformer (TF-IDF weighting)
- On these converted features, I trained a Logistic Regression classifier with balanced class weights to reduce bias toward frequent labels.

### Feature variants tested
- single words (1, 1)
- single words + word pairs (1, 2)
- Character n-grams (char_wb, 3–5)

Character n-grams performed better than complete words, probably because the job titles are multilingual and different spellings.

### Train / evaluation setup
- Training: label-list datasets (department-v2/seniority-v2)
- Final evaluation: LinkedIn titles (position) from the annotated dataset

### Evaluation outputs
For each run, I reported:
- Accuracy
- precision / recall / F1 (classification report)
- confusion matrix
Accuracy on the annotated LinkedIn set is dominated by errors in the catch-all classes (“Other” / “Professional”). Since the labels are imbalanced, I also use avg recall alongside accuracy for evaluating the models to better capture performance across all classes, including the minority ones.
### Ideas to increase recall for catch-all/unseen labels
**Inference process: Thresholds**

**Department ("Other" handling):**
- compute class probabilities
- if max probability < 0.40, overwrite prediction to "Other"

**Seniority (Professional handling):**
- compute class probabilities
- if max probability < 0.50, overwrite prediction to "Professional"
- additionally experimented applying additonal class-specific threshold for Senior(label most predicted instead of Professional with high confidence)

### Experiments to improve low recall of "Other"
Because "Other" had especially low recall, I tested two different strategies to increase recall of "Other":

**Two-stage model ("Other" vs Rest + Model for rest of labels)**
- Stage 1: classifier predicts "Other" vs non-"Other"
- Stage 2: only for non-"Other", classifier predicts the specific department
- Motivation: learn a stable boundary for "Other"

**Downsampling label-list training data**
- Limit the number of training examples per label (max 70 per department)
- Motivation: reduce dominance of large classes in department-v2 so the classifier does not overfit frequent labels and can improve minority-class performance

#Results

##Department

###single words
- department-v2 (full) + TF-IDF words (1, 1) + Logistic Regression
  - Acc.: 0.2635, Recall avg: 0.41
- department-v2 (full) + TF-IDF words (1, 1) + LR + confidence threshold (0.40) -> "Other"
  - Acc.: 0.6657, Recall avg: 0.45
  #
###word and word pairs
- department-v2 (full) + TF-IDF words and pairs of words (1, 2) + LR
  - Acc.: 0.2684, Recall avg: 0.42
- department-v2 (full) + TF-IDF words and pairs of words (1, 2) + LR + confidence threshold (0.40) -> "Other"
  - Acc.: 0.6615, Recall avg: 0.44

-> single words and word pairs are weaker than characters, probably caused by multilingual dataset. Character n-grams capture shared stems and tolerate spelling variants.
  #
###characters
- department-v2 (full) + TF-IDF character n-grams (3, 5) + LR
  - Acc.: 0.3302, Recall avg: 0.47
- department-v2 (full) + TF-IDF character n-grams (3, 5) + LR + confidence threshold (0.40) -> "Other"
  - Acc.: 0.6660, Recall avg: 0.48
- department-v2 (70/30 split) + TF-ID character n-grams (3, 5) + LR
  - Acc.: 0.3343, Recall avg: 0.48
  - department-v2 test-split: Acc.: 0.9137, Recall avg: 0.92
- **department-v2 (70/30 split) + TF-ID character n-grams (3, 5) + LR + confidence threshold (0.40) -> "Other"**
  - **Acc.: 0.6732, Recall avg: 0.46**
#

###other experiments
- department-v2 (full) + TF-IDF character n-grams (3, 5), words and word pairs (1, 2) + Two-stage LR (Other vs Rest, Calssify Rest)
  - Acc.: 0.3317 Recall avg: 0.48
- department-v2 (downsampled max 70/label) + TF-IDF character n-grams (3, 5) + LR
  - Acc.: 0.3359, Recall avg: 0.50

Downsampling has best accuracy of models without threshold.
probably beacuse it reduces dominance of large labels in training, but results can vary by random seed; it may also reduce coverage of rare vocabulary → sometimes better accuracy, worse Recall avg.

learning a better seperation between "Other" and rest is not possible through clear stages -> “Other” is not a coherent semantic class but rather a residual category (“everything else”), so Stage 1 cannot learn a stable boundary.
#

---


## Seniority
###single words
- seniority-v2 (full) + TF-IDF over words (1, 1) + Logistic Regression
  - Acc.: 0.3412, Recall avg: 0.54
- seniority-v2 (full) + TF-IDF over words (1, 1) + Logistic Regression + confidence threshold (0.50)
  - Acc.: 0.5470, Recall avg: 0.58
#
###word and word pairs
- seniority-v2 (full) + TF-IDF over words and pairs of words (1, 2) + Logistic Regression
  - Acc.: 0.3347, Recall avg: 0.53
- seniority-v2 (full) + TF-IDF over words and pairs of words (1, 2) + Logistic Regression
  - Acc.: 0.5489, Recall avg: 0.57

-> in seniority too, characters are stronger than complete words and word pairs
#
###characters
- seniority-v2 (full) + TF-IDF over character n-grams (char\_wb, 3, 5) + Logistic Regression
  - Acc.: 0.3582, Recall avg: 0.56
- **seniority-v2 (full) + TF-IDF char\_wb (3, 5) + LR + single confidence threshold (0.50)**
  - **Acc.: 0.5679, Recall avg: 0.60**
- seniority-v2 (full) + TF-IDF char\_wb (3, 5) + LR + tailored thresholds (0.50; Senior 0.97)
  - Acc.: 0.6497, Recall avg: 0.59
#

---
Tradeoff: thresholding usually increases Other recall and overall accuracy, but can lower precision/recall for smaller departments because borderline cases get absorbed into “Other”.
Also learning best threshold values would be tuning to the test data.
The same is true for the model with the three additional thresholds that are tuned to the eval on the annotated linkedIn data.


---
#

  All numbers are evaluation on LinkedIn (annotated) unless stated otherwise.


#Models

#Department

###**single words**

In [6]:
X_train = df_department["text"]
y_train = df_department["label"]

X_test = df_t_annotated["position"]
y_test = df_t_annotated["department"]

clf_dept_1words = Pipeline([
    ("vect", CountVectorizer(ngram_range=(1, 1))),
    ("tfidf", TfidfTransformer()),
    ("clf", LogisticRegression(class_weight="balanced",)),
    ])
clf_dept_1words.fit(X_train, y_train)

y_pred = clf_dept_1words.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.2634571645185747
                        precision    recall  f1-score   support

        Administrative       0.27      0.10      0.14        84
  Business Development       0.38      0.44      0.41        78
            Consulting       0.91      0.50      0.64       195
      Customer Support       0.86      0.38      0.52        48
       Human Resources       0.86      0.53      0.65        70
Information Technology       0.30      0.36      0.33       312
             Marketing       0.45      0.47      0.46       133
                 Other       0.63      0.01      0.02      1252
    Project Management       0.11      0.70      0.19       175
            Purchasing       0.92      0.32      0.47        72
                 Sales       0.25      0.78      0.37       219

              accuracy                           0.26      2638
             macro avg       0.54      0.41      0.38      2638
          weighted avg       0.54      0.26      0.22      2638



pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,8,2,0,0,0,17,2,2,40,0,13
Business Development,0,34,0,0,1,4,3,0,24,0,12
Consulting,0,19,97,0,3,7,2,0,54,0,13
Customer Support,1,1,0,18,0,7,0,0,11,0,10
Human Resources,1,0,2,0,37,3,3,0,13,0,11
Information Technology,0,8,0,3,0,111,2,3,141,1,43
Marketing,1,1,3,0,0,18,62,1,36,0,11
Other,16,20,1,0,2,164,48,12,609,1,379
Project Management,0,4,2,0,0,24,2,0,122,0,21


####with confidence threshold

In [7]:
probabilities = clf_dept_1words.predict_proba(X_test)
best_idx = probabilities.argmax(axis=1)
best_p = probabilities.max(axis=1)
pred = clf_dept_1words.classes_[best_idx]

THRESHOLD = 0.40

pred = pd.Series(pred)
pred[best_p < THRESHOLD] = "Other"
y_pred = pred.to_numpy()

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.66565579984837
                        precision    recall  f1-score   support

        Administrative       0.42      0.10      0.16        84
  Business Development       0.39      0.44      0.41        78
            Consulting       0.92      0.49      0.64       195
      Customer Support       0.89      0.35      0.51        48
       Human Resources       0.86      0.53      0.65        70
Information Technology       0.73      0.18      0.29       312
             Marketing       0.74      0.36      0.48       133
                 Other       0.62      0.95      0.75      1252
    Project Management       0.77      0.55      0.64       175
            Purchasing       0.96      0.32      0.48        72
                 Sales       0.94      0.67      0.78       219

              accuracy                           0.67      2638
             macro avg       0.75      0.45      0.53      2638
          weighted avg       0.71      0.67      0.63      2638



pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,8,2,0,0,0,0,0,74,0,0,0
Business Development,0,34,0,0,1,1,0,39,0,0,3
Consulting,0,19,96,0,3,2,2,71,1,0,1
Customer Support,1,1,0,17,0,0,0,28,0,0,1
Human Resources,0,0,1,0,37,1,0,31,0,0,0
Information Technology,0,7,0,2,0,56,0,231,16,0,0
Marketing,1,1,3,0,0,9,48,69,1,0,1
Other,7,19,1,0,2,4,12,1194,9,1,3
Project Management,0,4,1,0,0,3,0,69,97,0,1


###**words and word pairs**

In [8]:
X_train = df_department["text"]
y_train = df_department["label"]

X_test = df_t_annotated["position"]
y_test = df_t_annotated["department"]

clf_dept_words = Pipeline([
    ("vect", CountVectorizer(ngram_range=(1, 2))),
    ("tfidf", TfidfTransformer()),
    ("clf", LogisticRegression(class_weight="balanced",)),
    ])
clf_dept_words.fit(X_train, y_train)

y_pred = clf_dept_words.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.26838514025777105
                        precision    recall  f1-score   support

        Administrative       0.28      0.11      0.16        84
  Business Development       0.38      0.45      0.41        78
            Consulting       0.84      0.51      0.63       195
      Customer Support       0.89      0.35      0.51        48
       Human Resources       0.86      0.53      0.65        70
Information Technology       0.36      0.36      0.36       312
             Marketing       0.35      0.52      0.42       133
                 Other       0.67      0.01      0.02      1252
    Project Management       0.35      0.59      0.44       175
            Purchasing       0.96      0.32      0.48        72
                 Sales       0.13      0.87      0.22       219

              accuracy                           0.27      2638
             macro avg       0.55      0.42      0.39      2638
          weighted avg       0.56      0.27      0.22      2638



pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,9,2,0,0,0,13,9,2,5,0,44
Business Development,0,35,0,0,1,4,4,0,6,0,28
Consulting,0,18,99,0,3,10,2,0,22,0,41
Customer Support,1,1,0,17,0,4,1,0,0,0,24
Human Resources,1,0,5,0,37,3,5,0,1,0,18
Information Technology,0,8,1,2,0,113,6,2,63,0,117
Marketing,1,1,3,0,0,14,69,1,1,0,43
Other,16,21,5,0,2,109,78,12,79,1,929
Project Management,0,4,3,0,0,30,5,1,103,0,29


####with confidence threshold

In [9]:
probabilities = clf_dept_words.predict_proba(X_test)
best_idx = probabilities.argmax(axis=1)
best_p = probabilities.max(axis=1)
pred = clf_dept_words.classes_[best_idx]

THRESHOLD = 0.40

pred = pd.Series(pred)
pred[best_p < THRESHOLD] = "Other"
y_pred = pred.to_numpy()

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.6614859742228961
                        precision    recall  f1-score   support

        Administrative       0.40      0.10      0.15        84
  Business Development       0.38      0.44      0.41        78
            Consulting       0.92      0.49      0.64       195
      Customer Support       0.89      0.35      0.51        48
       Human Resources       0.88      0.53      0.66        70
Information Technology       0.74      0.18      0.29       312
             Marketing       0.72      0.35      0.47       133
                 Other       0.61      0.95      0.75      1252
    Project Management       0.92      0.55      0.69       175
            Purchasing       0.96      0.32      0.48        72
                 Sales       0.93      0.63      0.75       219

              accuracy                           0.66      2638
             macro avg       0.76      0.44      0.53      2638
          weighted avg       0.71      0.66      0.62      2638



pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,8,2,0,0,0,0,0,74,0,0,0
Business Development,0,34,0,0,0,1,0,40,0,0,3
Consulting,0,18,96,0,3,3,2,71,1,0,1
Customer Support,1,1,0,17,0,0,0,28,0,0,1
Human Resources,0,0,1,0,37,1,0,30,0,0,1
Information Technology,0,7,0,2,0,56,0,245,2,0,0
Marketing,1,1,3,0,0,8,46,73,0,0,1
Other,8,20,1,0,2,4,13,1195,5,1,3
Project Management,0,4,1,0,0,2,0,72,96,0,0


#####accuracy without "other" label in y_test

In [10]:
mask = (y_test != "Other")
print("Accuracy (non-Other only):", accuracy_score(y_test[mask], y_pred[mask]))
print(classification_report(y_test[mask], y_pred[mask], zero_division=0))


Accuracy (non-Other only): 0.3968253968253968
                        precision    recall  f1-score   support

        Administrative       0.67      0.10      0.17        84
  Business Development       0.49      0.44      0.46        78
            Consulting       0.93      0.49      0.64       195
      Customer Support       0.89      0.35      0.51        48
       Human Resources       0.93      0.53      0.67        70
Information Technology       0.78      0.18      0.29       312
             Marketing       0.90      0.35      0.50       133
                 Other       0.00      0.00      0.00         0
    Project Management       0.97      0.55      0.70       175
            Purchasing       1.00      0.32      0.48        72
                 Sales       0.95      0.63      0.75       219

              accuracy                           0.40      1386
             macro avg       0.77      0.36      0.47      1386
          weighted avg       0.86      0.40      0.52   

###**characters**

In [11]:
X_train = df_department["text"]
y_train = df_department["label"]

X_test = df_t_annotated["position"]
y_test = df_t_annotated["department"]

clf_dept_char = Pipeline([
    ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3,5))),
    ("tfidf", TfidfTransformer()),
    ("clf", LogisticRegression(class_weight="balanced")),
    ])
clf_dept_char.fit(X_train, y_train)

y_pred = clf_dept_char.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.33017437452615617
                        precision    recall  f1-score   support

        Administrative       0.13      0.18      0.15        84
  Business Development       0.40      0.45      0.42        78
            Consulting       0.85      0.54      0.66       195
      Customer Support       0.74      0.35      0.48        48
       Human Resources       0.78      0.56      0.65        70
Information Technology       0.17      0.90      0.28       312
             Marketing       0.57      0.43      0.49       133
                 Other       0.67      0.02      0.03      1252
    Project Management       0.72      0.64      0.68       175
            Purchasing       0.93      0.35      0.51        72
                 Sales       0.71      0.74      0.73       219

              accuracy                           0.33      2638
             macro avg       0.61      0.47      0.46      2638
          weighted avg       0.61      0.33      0.28      2638



pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,15,2,0,0,1,63,0,2,1,0,0
Business Development,0,35,0,0,1,34,1,3,0,0,4
Consulting,0,18,106,0,5,48,8,0,7,0,3
Customer Support,2,1,1,17,0,26,0,0,0,0,1
Human Resources,1,0,4,0,39,23,0,0,0,0,3
Information Technology,0,7,0,3,0,282,0,3,7,0,10
Marketing,4,1,4,0,0,63,57,1,2,0,1
Other,89,20,6,3,1,1019,29,20,23,2,40
Project Management,0,4,1,0,3,52,0,1,112,0,2


####with confidence threshold

In [12]:
probabilities = clf_dept_char.predict_proba(X_test)
best_idx = probabilities.argmax(axis=1)
best_p = probabilities.max(axis=1)
pred = clf_dept_char.classes_[best_idx]

THRESHOLD = 0.40

pred = pd.Series(pred)
pred[best_p < THRESHOLD] = "Other"
y_pred = pred.to_numpy()

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.6660348749052313
                        precision    recall  f1-score   support

        Administrative       0.45      0.15      0.23        84
  Business Development       0.40      0.45      0.42        78
            Consulting       0.90      0.52      0.66       195
      Customer Support       0.89      0.35      0.51        48
       Human Resources       0.81      0.54      0.65        70
Information Technology       0.42      0.36      0.39       312
             Marketing       0.69      0.40      0.50       133
                 Other       0.65      0.88      0.75      1252
    Project Management       0.89      0.63      0.74       175
            Purchasing       0.96      0.35      0.51        72
                 Sales       0.90      0.70      0.79       219

              accuracy                           0.67      2638
             macro avg       0.73      0.48      0.56      2638
          weighted avg       0.69      0.67      0.65      2638



pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,13,2,0,0,1,13,0,55,0,0,0
Business Development,0,35,0,0,1,2,1,36,0,0,3
Consulting,0,18,102,0,3,3,3,63,2,0,1
Customer Support,1,1,1,17,0,6,0,22,0,0,0
Human Resources,0,0,1,0,38,1,0,29,0,0,1
Information Technology,0,7,0,2,0,112,0,187,3,0,1
Marketing,3,1,3,0,0,16,53,56,0,0,1
Other,9,19,2,0,1,91,15,1099,6,1,9
Project Management,0,4,1,0,3,13,0,43,110,0,1


#####accuracy without "other" label in y_test

In [13]:
mask = (y_test != "Other")
print("Accuracy (non-Other only):", accuracy_score(y_test[mask], y_pred[mask]))
print(classification_report(y_test[mask], y_pred[mask], zero_division=0))


Accuracy (non-Other only): 0.47474747474747475
                        precision    recall  f1-score   support

        Administrative       0.65      0.15      0.25        84
  Business Development       0.51      0.45      0.48        78
            Consulting       0.92      0.52      0.67       195
      Customer Support       0.89      0.35      0.51        48
       Human Resources       0.83      0.54      0.66        70
Information Technology       0.64      0.36      0.46       312
             Marketing       0.85      0.40      0.54       133
                 Other       0.00      0.00      0.00         0
    Project Management       0.94      0.63      0.75       175
            Purchasing       1.00      0.35      0.52        72
                 Sales       0.95      0.70      0.81       219

              accuracy                           0.47      1386
             macro avg       0.74      0.41      0.51      1386
          weighted avg       0.82      0.47      0.59  

#####find best threshold

In [15]:
for i in np.arange(0.30, 0.65, 0.05):
    y_pred = pred.copy()
    y_pred[best_p < i] = "Other"

    print(f"Accuracy (THRESHOLD={i:.2f}):", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, zero_division=0))

Accuracy (THRESHOLD=0.30): 0.6660348749052313
                        precision    recall  f1-score   support

        Administrative       0.45      0.15      0.23        84
  Business Development       0.40      0.45      0.42        78
            Consulting       0.90      0.52      0.66       195
      Customer Support       0.89      0.35      0.51        48
       Human Resources       0.81      0.54      0.65        70
Information Technology       0.42      0.36      0.39       312
             Marketing       0.69      0.40      0.50       133
                 Other       0.65      0.88      0.75      1252
    Project Management       0.89      0.63      0.74       175
            Purchasing       0.96      0.35      0.51        72
                 Sales       0.90      0.70      0.79       219

              accuracy                           0.67      2638
             macro avg       0.73      0.48      0.56      2638
          weighted avg       0.69      0.67      0.65   

###**characters with trainingsplit (70/30) of department-v2**

In [16]:
X_all = df_department["text"]
y_all = df_department["label"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X_all, y_all, test_size=0.70, random_state=42, stratify=y_all
)

clf_dept_char_split = Pipeline([
    ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3, 5))),
    ("tfidf", TfidfTransformer()),
    ("clf", LogisticRegression(class_weight="balanced")),
    ])

clf_dept_char_split.fit(X_train, y_train)

# eval on department-v2-test-split
y_pred_2 = clf_dept_char_split.predict(X_eval)

print("test-split Accuracy:", accuracy_score(y_eval, y_pred_2))
print(classification_report(y_eval, y_pred_2, zero_division=0))
labels = sorted(set(y_eval) | set(y_eval))
cm = confusion_matrix(y_eval, y_pred_2, labels=labels)
cm_df_2 = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
display(cm_df_2)

# eval on LinkedIn annotated

X_test = df_t_annotated["position"]
y_test = df_t_annotated["department"].to_numpy()

# --- without threshold ---
y_pred_2 = clf_dept_char_split.predict(X_test)

print("LinkedIn Accuracy (without threshold):", accuracy_score(y_test, y_pred_2))
print(classification_report(y_test, y_pred_2, zero_division=0))
labels = sorted(set(y_test) | set(y_pred_2))
cm = confusion_matrix(y_test, y_pred_2, labels=labels)
cm_df_2 = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
display(cm_df_2)


test-split Accuracy: 0.9136862855533653
                        precision    recall  f1-score   support

        Administrative       0.46      0.93      0.62        58
  Business Development       0.74      0.98      0.84       434
            Consulting       0.56      0.97      0.71       117
      Customer Support       0.40      0.91      0.56        23
       Human Resources       0.80      0.73      0.76        22
Information Technology       0.93      0.88      0.90       913
             Marketing       0.99      0.94      0.96      3007
                 Other       0.42      1.00      0.59        29
    Project Management       0.52      0.96      0.67       141
            Purchasing       0.65      1.00      0.79        28
                 Sales       0.99      0.88      0.93      2330

              accuracy                           0.91      7102
             macro avg       0.68      0.92      0.76      7102
          weighted avg       0.94      0.91      0.92      710

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,54,0,0,0,0,1,1,0,2,0,0
Business Development,0,427,1,0,0,1,0,1,3,0,1
Consulting,0,0,113,0,0,2,0,0,2,0,0
Customer Support,0,0,0,21,0,0,0,0,1,0,1
Human Resources,1,0,1,0,16,2,1,0,1,0,0
Information Technology,4,1,25,6,1,800,3,15,51,1,6
Marketing,33,52,19,5,2,17,2816,8,43,5,7
Other,0,0,0,0,0,0,0,29,0,0,0
Project Management,0,0,1,0,0,4,0,0,135,0,1


LinkedIn Accuracy (without threshold): 0.33434420015163
                        precision    recall  f1-score   support

        Administrative       0.15      0.15      0.15        84
  Business Development       0.40      0.47      0.44        78
            Consulting       0.86      0.55      0.67       195
      Customer Support       0.76      0.40      0.52        48
       Human Resources       0.51      0.57      0.54        70
Information Technology       0.17      0.90      0.29       312
             Marketing       0.53      0.45      0.49       133
                 Other       0.69      0.02      0.04      1252
    Project Management       0.77      0.63      0.70       175
            Purchasing       0.87      0.38      0.52        72
                 Sales       0.61      0.74      0.67       219

              accuracy                           0.33      2638
             macro avg       0.57      0.48      0.46      2638
          weighted avg       0.61      0.33   

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,13,2,0,0,1,63,0,2,1,0,2
Business Development,0,37,0,0,1,27,6,3,2,0,2
Consulting,0,18,108,0,9,46,8,0,5,0,1
Customer Support,1,1,1,19,0,24,0,0,0,0,2
Human Resources,1,0,3,0,40,22,0,0,0,0,4
Information Technology,0,7,0,2,1,281,0,4,7,0,10
Marketing,3,1,4,0,1,61,60,1,1,0,1
Other,66,20,6,4,22,984,34,24,11,4,77
Project Management,0,4,1,0,4,52,0,1,111,0,2


In [17]:
probabilities = clf_dept_char_split.predict_proba(X_test)
best_idx = probabilities.argmax(axis=1)
best_p = probabilities.max(axis=1)
pred = clf_dept_char_split.classes_[best_idx]

THRESHOLD = 0.40

pred = pd.Series(pred)
pred[best_p < THRESHOLD] = "Other"
y_pred = pred.to_numpy()

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.6732373009855952
                        precision    recall  f1-score   support

        Administrative       0.50      0.12      0.19        84
  Business Development       0.40      0.46      0.43        78
            Consulting       0.88      0.50      0.64       195
      Customer Support       0.89      0.33      0.48        48
       Human Resources       0.80      0.40      0.53        70
Information Technology       0.65      0.29      0.41       312
             Marketing       0.67      0.38      0.49       133
                 Other       0.63      0.94      0.76      1252
    Project Management       0.89      0.61      0.72       175
            Purchasing       0.96      0.32      0.48        72
                 Sales       0.93      0.65      0.77       219

              accuracy                           0.67      2638
             macro avg       0.75      0.46      0.54      2638
          weighted avg       0.71      0.67      0.65      2638



pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,10,2,0,0,1,8,0,63,0,0,0
Business Development,0,36,0,0,0,1,0,40,0,0,1
Consulting,0,18,98,0,3,2,3,68,2,0,1
Customer Support,1,1,1,16,0,3,0,26,0,0,0
Human Resources,0,0,2,0,28,1,0,39,0,0,0
Information Technology,0,7,0,2,0,92,0,207,4,0,0
Marketing,1,1,4,0,0,9,51,66,0,0,1
Other,6,19,2,0,0,19,18,1174,6,1,7
Project Management,0,4,1,0,3,5,0,56,106,0,0


###**characters two stage**

In [18]:
OTHER = "Other"

# Stage 1: Other vs Rest
y_train_is_other = (y_train == OTHER).astype(int)
y_test_is_other  = (y_test  == OTHER).astype(int)


other_clf = Pipeline([
    ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3, 5))),
    ("tfidf", TfidfTransformer()),
    ("clf", LogisticRegression(class_weight="balanced"))
    ])
other_clf.fit(X_train, y_train_is_other)


stage_1_pred = other_clf.predict(X_test).astype(int)

# Stage 2 - training only with !="other"

mask_train_not_other = (y_train != OTHER)

dept_clf = Pipeline([
    ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3, 5))),
    ("tfidf", TfidfTransformer()),
    ("clf", LogisticRegression(class_weight="balanced"))
    ])
dept_clf.fit(X_train[mask_train_not_other], y_train[mask_train_not_other])

# predict final: Stage 1 runs on all test rows, but its output is ignored wherever Stage 1 says “Other”
stage_2_pred = dept_clf.predict(X_test)
y_pred = np.where(stage_1_pred == 1, OTHER, stage_2_pred)


print("Stage 1 (Other vs Rest) accuracy:", accuracy_score(y_test_is_other, stage_1_pred))
print(classification_report(y_test_is_other, stage_1_pred, target_names=["Not-Other", "Other"], zero_division=0))

mask_test_not_other = (y_test != OTHER)
print("Stage 2 accuracy (on not-Other rows):", accuracy_score(y_test[mask_test_not_other], dept_clf.predict(X_test[mask_test_not_other])))
print(classification_report(y_test[mask_test_not_other], dept_clf.predict(X_test[mask_test_not_other]), zero_division=0))

print("Overall accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))


labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Stage 1 (Other vs Rest) accuracy: 0.5272934040940106
              precision    recall  f1-score   support

   Not-Other       0.53      0.99      0.69      1386
       Other       0.60      0.01      0.02      1252

    accuracy                           0.53      2638
   macro avg       0.56      0.50      0.36      2638
weighted avg       0.56      0.53      0.37      2638

Stage 2 accuracy (on not-Other rows): 0.6233766233766234
                        precision    recall  f1-score   support

        Administrative       0.62      0.15      0.25        84
  Business Development       0.51      0.47      0.49        78
            Consulting       0.89      0.56      0.69       195
      Customer Support       0.90      0.40      0.55        48
       Human Resources       0.70      0.57      0.63        70
Information Technology       0.43      0.92      0.59       312
             Marketing       0.75      0.45      0.56       133
    Project Management       0.82      0.62      0

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,13,2,0,0,1,62,0,2,2,0,2
Business Development,0,37,0,0,1,28,6,2,2,0,2
Consulting,0,18,109,0,8,46,8,0,5,0,1
Customer Support,1,1,1,19,0,23,0,0,0,0,3
Human Resources,1,0,4,0,40,20,0,0,0,0,5
Information Technology,0,7,0,2,1,285,1,3,7,0,6
Marketing,3,1,4,0,1,60,59,2,2,0,1
Other,65,19,6,4,22,1005,34,15,11,4,67
Project Management,0,4,1,0,4,53,0,1,109,0,3


###**characters downsample**

In [19]:
# downsample department label list to max n rows per label

df_train = df_department.copy()

N_PER_LABEL = 70

df_train_n = (
    df_train.groupby("label", group_keys=False)
            .apply(lambda g: g.sample(n=min(len(g), N_PER_LABEL), random_state=42))
            .reset_index(drop=True)
)

print("Training size (downsampled):", len(df_train_n))
print(df_train_n["label"].value_counts())


df_test = df_t_annotated.copy()
df_test["text"] = df_test["position"]
df_test["label"] = df_test["department"]


X_train = df_train_n["text"]
y_train = df_train_n["label"]
X_test  = df_test["text"]
y_test  = df_test["label"]

clf = Pipeline([
    ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3, 5))),
    ("tfidf", TfidfTransformer()),
    ("clf", LogisticRegression(class_weight="balanced"))
    ])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

/tmp/ipython-input-1409469061.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(len(g), N_PER_LABEL), random_state=42))


Training size (downsampled): 636
label
Administrative            70
Business Development      70
Consulting                70
Marketing                 70
Information Technology    70
Sales                     70
Project Management        70
Other                     42
Purchasing                40
Customer Support          33
Human Resources           31
Name: count, dtype: int64
Accuracy: 0.33586050037907506
                        precision    recall  f1-score   support

        Administrative       0.10      0.17      0.12        84
  Business Development       0.33      0.49      0.40        78
            Consulting       0.69      0.61      0.64       195
      Customer Support       0.43      0.42      0.43        48
       Human Resources       0.25      0.73      0.37        70
Information Technology       0.18      0.81      0.30       312
             Marketing       0.54      0.44      0.48       133
                 Other       0.62      0.02      0.04      1252
    Proje

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Other,Project Management,Purchasing,Sales
true,,,,,,,,,,,
Administrative,14,2,0,0,11,47,1,2,3,3,1
Business Development,0,38,14,1,6,13,1,4,0,0,1
Consulting,0,14,118,0,14,36,8,0,4,0,1
Customer Support,2,1,2,20,1,22,0,0,0,0,0
Human Resources,1,1,4,0,51,12,0,0,0,0,1
Information Technology,1,26,2,3,9,254,2,4,6,0,5
Marketing,4,1,4,0,5,54,58,3,3,0,1
Other,119,25,23,20,98,826,31,25,36,8,41
Project Management,0,4,1,0,4,34,1,2,129,0,0


#Seniority

###**single words**

In [20]:
X_train = df_seniority["text"]
y_train = df_seniority["label"]

X_test = df_t_annotated["position"]
y_test = df_t_annotated["seniority"]

clf_sen_1words = Pipeline([
    ("vect", CountVectorizer(ngram_range=(1, 1))),
    ("tfidf", TfidfTransformer()),
    ("clf", LogisticRegression(class_weight="balanced",)),
    ])
clf_sen_1words.fit(X_train, y_train)

y_pred = clf_sen_1words.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.3411675511751327
              precision    recall  f1-score   support

    Director       0.73      0.85      0.78       141
      Junior       0.37      0.19      0.25       230
        Lead       0.27      0.74      0.39       460
  Management       0.82      0.60      0.69       418
Professional       0.00      0.00      0.00      1219
      Senior       0.19      0.86      0.31       170

    accuracy                           0.34      2638
   macro avg       0.40      0.54      0.40      2638
weighted avg       0.26      0.34      0.26      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,120,0,15,3,0,3
Junior,1,43,134,3,0,49
Lead,2,7,340,10,0,101
Management,40,2,85,250,0,41
Professional,0,62,688,37,0,432
Senior,2,2,18,1,0,147


####with confidence threshold

In [21]:
probabilities = clf_sen_1words.predict_proba(X_test)
best_idx = probabilities.argmax(axis=1)
best_p = probabilities.max(axis=1)
pred = clf_sen_1words.classes_[best_idx]

THRESHOLD = 0.50
pred = pd.Series(pred)
pred[best_p < THRESHOLD] = "Professional"
y_pred = pred.to_numpy()

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.5470053070507961
              precision    recall  f1-score   support

    Director       0.74      0.85      0.79       141
      Junior       0.39      0.19      0.25       230
        Lead       0.71      0.51      0.59       460
  Management       0.88      0.55      0.68       418
Professional       0.63      0.55      0.59      1219
      Senior       0.20      0.86      0.33       170

    accuracy                           0.55      2638
   macro avg       0.59      0.58      0.54      2638
weighted avg       0.64      0.55      0.57      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,120,0,1,0,17,3
Junior,1,43,7,2,129,48
Lead,1,7,233,8,125,86
Management,39,1,12,231,101,34
Professional,0,57,69,20,670,403
Senior,2,2,4,1,15,146


###**words and word pairs**

In [22]:
X_train = df_seniority["text"]
y_train = df_seniority["label"]

X_test = df_t_annotated["position"]
y_test = df_t_annotated["seniority"]

clf_sen_words = Pipeline([
    ("vect", CountVectorizer(ngram_range=(1, 2))),
    ("tfidf", TfidfTransformer()),
    ("clf", LogisticRegression(class_weight="balanced",)),
    ])
clf_sen_words.fit(X_train, y_train)

y_pred = clf_sen_words.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.33472327520849127
              precision    recall  f1-score   support

    Director       0.72      0.86      0.78       141
      Junior       0.33      0.17      0.23       230
        Lead       0.27      0.72      0.39       460
  Management       0.85      0.60      0.70       418
Professional       0.00      0.00      0.00      1219
      Senior       0.17      0.83      0.28       170

    accuracy                           0.33      2638
   macro avg       0.39      0.53      0.40      2638
weighted avg       0.26      0.33      0.26      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,121,0,14,2,0,4
Junior,1,40,133,3,0,53
Lead,2,7,332,9,0,110
Management,42,2,82,249,0,43
Professional,1,63,654,27,0,474
Senior,2,8,17,2,0,141


####with confidence threshold

In [23]:
probabilities = clf_sen_words.predict_proba(X_test)
best_idx = probabilities.argmax(axis=1)
best_p = probabilities.max(axis=1)
pred = clf_sen_words.classes_[best_idx]

THRESHOLD = 0.50
pred = pd.Series(pred)
pred[best_p < THRESHOLD] = "Professional"
y_pred = pred.to_numpy()

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.5489006823351024
              precision    recall  f1-score   support

    Director       0.73      0.85      0.79       141
      Junior       0.36      0.17      0.23       230
        Lead       0.79      0.49      0.61       460
  Management       0.89      0.54      0.67       418
Professional       0.63      0.57      0.60      1219
      Senior       0.19      0.81      0.31       170

    accuracy                           0.55      2638
   macro avg       0.60      0.57      0.53      2638
weighted avg       0.65      0.55      0.57      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,120,0,0,0,18,3
Junior,1,40,5,3,129,52
Lead,1,7,226,8,135,83
Management,40,0,12,225,108,33
Professional,0,56,42,16,699,406
Senior,2,8,2,1,19,138


#####accuracy without "Professional" label in y_test

In [24]:
mask = (y_test != "Professional")
print("Accuracy (non-Professional only):", accuracy_score(y_test[mask], y_pred[mask]))
print(classification_report(y_test[mask], y_pred[mask], zero_division=0))


Accuracy (non-Professional only): 0.5278365045806906
              precision    recall  f1-score   support

    Director       0.73      0.85      0.79       141
      Junior       0.73      0.17      0.28       230
        Lead       0.92      0.49      0.64       460
  Management       0.95      0.54      0.69       418
Professional       0.00      0.00      0.00         0
      Senior       0.45      0.81      0.58       170

    accuracy                           0.53      1419
   macro avg       0.63      0.48      0.50      1419
weighted avg       0.82      0.53      0.60      1419



###**characters**

In [25]:
X_train = df_seniority["text"]
y_train = df_seniority["label"]

X_test = df_t_annotated["position"]
y_test = df_t_annotated["seniority"]

clf_sen_char = Pipeline([
    ("vect", CountVectorizer(analyzer="char_wb", ngram_range=(3,5),)),
    ("tfidf", TfidfTransformer()),
    ("clf", LogisticRegression(class_weight="balanced",)),
    ])
clf_sen_char.fit(X_train, y_train)

y_pred = clf_sen_char.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.3582259287338893
              precision    recall  f1-score   support

    Director       0.70      0.92      0.80       141
      Junior       0.33      0.22      0.26       230
        Lead       0.45      0.68      0.54       460
  Management       0.47      0.74      0.57       418
Professional       0.00      0.00      0.00      1219
      Senior       0.15      0.82      0.25       170

    accuracy                           0.36      2638
   macro avg       0.35      0.56      0.40      2638
weighted avg       0.23      0.36      0.27      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,130,0,5,3,0,3
Junior,1,50,41,44,0,94
Lead,3,7,315,22,0,113
Management,43,2,23,310,0,40
Professional,6,82,316,273,0,542
Senior,3,11,3,13,0,140


#####accuracy without "Professional" label in y_test

In [26]:
mask = (y_test != "Professional")
print("Accuracy (non-Professional only):", accuracy_score(y_test[mask], y_pred[mask]))
print(classification_report(y_test[mask], y_pred[mask], zero_division=0))

Accuracy (non-Professional only): 0.6659619450317125
              precision    recall  f1-score   support

    Director       0.72      0.92      0.81       141
      Junior       0.71      0.22      0.33       230
        Lead       0.81      0.68      0.74       460
  Management       0.79      0.74      0.77       418
      Senior       0.36      0.82      0.50       170

    accuracy                           0.67      1419
   macro avg       0.68      0.68      0.63      1419
weighted avg       0.73      0.67      0.66      1419



####with confidence threshold

#####single confidence threshold

In [27]:
probabilities = clf_sen_char.predict_proba(X_test)
best_idx = probabilities.argmax(axis=1)
best_p = probabilities.max(axis=1)
pred = clf_sen_char.classes_[best_idx]

THRESHOLD = 0.50
pred = pd.Series(pred)
pred[best_p < THRESHOLD] = "Professional"
y_pred = pred.to_numpy()

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.5678544351781653
              precision    recall  f1-score   support

    Director       0.73      0.89      0.80       141
      Junior       0.36      0.20      0.26       230
        Lead       0.88      0.57      0.69       460
  Management       0.86      0.61      0.71       418
Professional       0.68      0.55      0.61      1219
      Senior       0.18      0.81      0.30       170

    accuracy                           0.57      2638
   macro avg       0.61      0.60      0.56      2638
weighted avg       0.69      0.57      0.60      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,125,0,1,1,11,3
Junior,1,46,4,3,125,51
Lead,1,6,260,5,90,98
Management,43,2,9,255,74,35
Professional,0,64,23,29,674,429
Senior,2,11,0,3,16,138


#####accuracy without "Professional" label in y_test

In [28]:
mask = (y_test != "Professional")
print("Accuracy (non-Professional only):", accuracy_score(y_test[mask], y_pred[mask]))
print(classification_report(y_test[mask], y_pred[mask], zero_division=0))


Accuracy (non-Professional only): 0.5806906272022551
              precision    recall  f1-score   support

    Director       0.73      0.89      0.80       141
      Junior       0.71      0.20      0.31       230
        Lead       0.95      0.57      0.71       460
  Management       0.96      0.61      0.74       418
Professional       0.00      0.00      0.00         0
      Senior       0.42      0.81      0.56       170

    accuracy                           0.58      1419
   macro avg       0.63      0.51      0.52      1419
weighted avg       0.83      0.58      0.65      1419



#####find best threshold

In [30]:
for i in np.arange(0.30, 0.65, 0.05):
    y_pred = pred.copy()
    y_pred[best_p < i] = "Other"

    print(f"Accuracy (THRESHOLD={i:.2f}):", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, zero_division=0))


Accuracy (THRESHOLD=0.30): 0.5284306292645944
              precision    recall  f1-score   support

    Director       0.73      0.89      0.80       141
      Junior       0.36      0.20      0.26       230
        Lead       0.88      0.57      0.69       460
  Management       0.86      0.61      0.71       418
       Other       0.00      0.00      0.00         0
Professional       0.67      0.47      0.55      1219
      Senior       0.18      0.81      0.30       170

    accuracy                           0.53      2638
   macro avg       0.52      0.51      0.47      2638
weighted avg       0.68      0.53      0.57      2638

Accuracy (THRESHOLD=0.35): 0.4241849886277483
              precision    recall  f1-score   support

    Director       0.73      0.89      0.80       141
      Junior       0.36      0.20      0.26       230
        Lead       0.88      0.57      0.69       460
  Management       0.86      0.61      0.71       418
       Other       0.00      0.00      0

#####Summary of probabilities for true “Professional” positions

In [31]:
y_pred = clf_sen_char.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))
cm_df

Accuracy: 0.3582259287338893
              precision    recall  f1-score   support

    Director       0.70      0.92      0.80       141
      Junior       0.33      0.22      0.26       230
        Lead       0.45      0.68      0.54       460
  Management       0.47      0.74      0.57       418
Professional       0.00      0.00      0.00      1219
      Senior       0.15      0.82      0.25       170

    accuracy                           0.36      2638
   macro avg       0.35      0.56      0.40      2638
weighted avg       0.23      0.36      0.27      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,130,0,5,3,0,3
Junior,1,50,41,44,0,94
Lead,3,7,315,22,0,113
Management,43,2,23,310,0,40
Professional,6,82,316,273,0,542
Senior,3,11,3,13,0,140


Most confused labels: lead, Managment, Senior

In [32]:
probabilities = clf_sen_char.predict_proba(X_test)
classes = clf_sen_char.named_steps["clf"].classes_
probabilities_df = pd.DataFrame(probabilities, columns=classes, index=X_test.index)

top1_idx = probabilities.argmax(axis=1)
pred = classes[top1_idx]
p_pred = probabilities.max(axis=1)


rows = pd.DataFrame({
    "position": X_test,
    "true": y_test,
    "pred": pred,
    "p_pred": p_pred,
}, index=X_test.index)

rows = rows.join(probabilities_df)
prof_rows = rows[rows["true"] == "Professional"]
pd.set_option("display.float_format", "{:.4f}".format)


conf_pct = (prof_rows["pred"].value_counts(normalize=True) * 100).round(2)
print("Confusion (Professional -> predicted as ...):\n", conf_pct)


print("\nConfidence by predicted label (true label = Professional):")
print(prof_rows.groupby("pred")["p_pred"].describe())


Confusion (Professional -> predicted as ...):
 pred
Senior       44.4600
Lead         25.9200
Management   22.4000
Junior        6.7300
Director      0.4900
Name: proportion, dtype: float64

Confidence by predicted label (true label = Professional):
              count   mean    std    min    25%    50%    75%    max
pred                                                                
Director     6.0000 0.3669 0.0576 0.3016 0.3194 0.3681 0.4184 0.4250
Junior      82.0000 0.7609 0.2275 0.2636 0.5265 0.8769 0.9236 0.9907
Lead       316.0000 0.3627 0.0830 0.2524 0.3123 0.3400 0.3789 0.7906
Management 273.0000 0.3884 0.1198 0.2395 0.3194 0.3512 0.4028 0.9284
Senior     542.0000 0.7757 0.2486 0.2563 0.6057 0.9035 0.9817 1.0000


75%

Senior -> very high confidence in wrong prediction 0.98

Lead, Managment -> confidence not as high 0.38/0.40 (below previous 0.50 threshold, so ignoring those)

#####additional individual thresholds for Senior

In [42]:
THRESHOLD = 0.50
THRESHOLD_SEN  = 0.97


probabilities = clf_sen_char.predict_proba(X_test)
classes = clf_sen_char.named_steps["clf"].classes_

best_idx = probabilities.argmax(axis=1)
best_p = probabilities.max(axis=1)
pred = classes[best_idx]

idx = {c: i for i, c in enumerate(classes)}

pS = probabilities[:, idx["Senior"]]



is_prof = (
    (best_p < THRESHOLD) |
    ((pred == "Senior") & (pS < THRESHOLD_SEN))
    )

y_pred = np.where(is_prof, "Professional", pred)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)
pd.DataFrame(cm, index=pd.Index(labels, name="true"), columns=pd.Index(labels, name="pred"))


Accuracy: 0.6497346474601972
              precision    recall  f1-score   support

    Director       0.73      0.89      0.80       141
      Junior       0.36      0.20      0.26       230
        Lead       0.88      0.57      0.69       460
  Management       0.86      0.61      0.71       418
Professional       0.64      0.77      0.70      1219
      Senior       0.30      0.50      0.38       170

    accuracy                           0.65      2638
   macro avg       0.63      0.59      0.59      2638
weighted avg       0.68      0.65      0.65      2638



pred,Director,Junior,Lead,Management,Professional,Senior
true,,,,,,
Director,125,0,1,1,14,0
Junior,1,46,4,3,174,2
Lead,1,6,260,5,171,17
Management,43,2,9,255,93,16
Professional,0,64,23,29,943,160
Senior,2,11,0,3,69,85
